# Phase 1: Data Preparation & Validation

This phase focuses on converting unstructured election documents into structured datasets. Because real-world election data is often messy, we implemented a rigorous cleaning and validation workflow.

### Key Accomplishments:
* **Data Extraction**: Processed raw **ECT Form 5/18** (Constituency) and **Form 5/18 บช** (Party-list) into tabular format.
* **Mathematical Validation**: Verified data integrity using the official ballot balance formula:
$$\text{ballots\_used} = \text{ballots\_good} + \text{ballots\_spoiled} + \text{ballots\_no\_vote}$$
* **Missing Value Recovery**: 
    * Recovered `ballots_good` by cross-referencing individual candidate/party vote sums.
    * Imputed `eligible_voters` using a **71% national turnout baseline** to ensure a complete dataset for analysis.
* **Standardization**: Unified the party-list schema by merging `party_name` into the `entity_name` column.
* **Pipeline Organization**: Structured the project to save validated outputs to `data/clean_data/` (external to the notebooks directory) for seamless integration with the **Streamlit** dashboard.

### Final Datasets:
1. `5_18_station.csv`: Polling station metadata and ballot statistics of 5/18.
2. `5_18_party_station.csv`: Polling station metadata and ballot statistics of 5/18 บช.
3. `5_18_votes.csv`: Candidate-level constituency results.
4. `5_18_party_vote.csv`: Party-level party-list results.


In [64]:
import pandas as pd
import numpy as np

In [65]:
stations_df = pd.read_csv("../data/processed/stations.csv")
votes_df = pd.read_csv("../data/processed/votes.csv")

In [66]:
col_to_drop = ['source_pdf', 'source_pages', 'ocr_status','ballots_received']
stations_df.drop(columns=col_to_drop, inplace=True)

In [67]:
stations_df.drop_duplicates(inplace=True)
stations_df.dropna(subset=['station_code'], inplace=True)

In [68]:
station = stations_df.copy()

In [ ]:
STATION_AUDIT_COLS = [
    'eligible_voters',
    'voters_present',
    'ballots_used',
    'ballots_good',
    'ballots_spoiled',
    'ballots_no_vote',
]


def normalize_station_code_series(series):
    return series.astype(str).str.strip().str.replace('?', '', regex=False)


def add_station_code_key(df):
    df = df.copy()
    if 'station_code' in df.columns:
        df['station_code_key'] = normalize_station_code_series(df['station_code'])
    return df


def append_validation_flag(df, mask, flag):
    if 'validation_flags' not in df.columns:
        df['validation_flags'] = ''
    existing = df.loc[mask, 'validation_flags'].fillna('').astype(str)
    df.loc[mask, 'validation_flags'] = existing.where(existing.eq(''), existing + '|') + flag
    return df


def ensure_station_audit_columns(df):
    df = add_station_code_key(df)
    if 'validation_flags' not in df.columns:
        df['validation_flags'] = ''
    for col in STATION_AUDIT_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            raw_col = f'{col}_raw'
            reason_col = f'{col}_correction_reason'
            corrected_col = f'{col}_corrected'
            if raw_col not in df.columns:
                df[raw_col] = df[col]
            if reason_col not in df.columns:
                df[reason_col] = ''
            df[corrected_col] = df[col]
    return df


def refresh_station_corrected_columns(df):
    for col in STATION_AUDIT_COLS:
        if col in df.columns:
            df[f'{col}_corrected'] = df[col]
    return df


def fill_missing_ballot_zero(df, cols):
    df = ensure_station_audit_columns(df)
    for col in cols:
        if col not in df.columns:
            continue
        mask = df[col].isna()
        if not mask.any():
            continue
        df.loc[mask, col] = 0
        df.loc[mask, f'{col}_correction_reason'] = 'filled_missing_with_zero_for_ballot_component'
        df = append_validation_flag(df, mask, f'{col}:filled_missing_with_zero')
    return refresh_station_corrected_columns(df)


def station_quality_sort(df):
    df = ensure_station_audit_columns(df)
    df['_station_code_uncertain'] = df['station_code'].astype(str).str.contains('?', regex=False)
    df['_validation_flag_count'] = (
        df['validation_flags'].fillna('').astype(str)
        .str.replace(';', '|', regex=False)
        .str.split('|')
        .map(lambda flags: sum(bool(flag.strip()) for flag in flags))
    )
    present_numeric = [col for col in STATION_AUDIT_COLS if col in df.columns]
    df['_numeric_completeness'] = df[present_numeric].notna().sum(axis=1) if present_numeric else 0
    sort_cols = ['station_code_key', '_station_code_uncertain', '_validation_flag_count', '_numeric_completeness']
    return df.sort_values(sort_cols, ascending=[True, True, True, False], kind='mergesort')


def select_best_station_rows(df):
    df = station_quality_sort(df)
    return (
        df.drop_duplicates(subset=['station_code_key'], keep='first')
        .drop(columns=['_station_code_uncertain', '_validation_flag_count', '_numeric_completeness'], errors='ignore')
        .reset_index(drop=True)
    )


def write_station_duplicate_review(df, output_path):
    review = station_quality_sort(df)
    review = review[review.duplicated('station_code_key', keep=False)].copy()
    if not review.empty:
        review['selected_for_clean_output'] = ~review.duplicated('station_code_key', keep='first')
    review = review.drop(columns=['_station_code_uncertain', '_validation_flag_count', '_numeric_completeness'], errors='ignore')
    review.to_csv(os.path.join(output_path, 'qa_duplicate_stations.csv'), index=False, encoding='utf-8-sig')
    return review


stations_df = fill_missing_ballot_zero(stations_df, ['ballots_no_vote', 'ballots_spoiled'])


In [ ]:
def heal_election_data(stations_df, votes_df):
    df = ensure_station_audit_columns(stations_df)

    votes_calc = add_station_code_key(votes_df)
    votes_calc['votes'] = pd.to_numeric(votes_calc['votes'], errors='coerce')
    votes_sum = (
        votes_calc.groupby('station_code_key', dropna=False)['votes']
        .sum(min_count=1)
        .reset_index(name='calculated_good_ballots')
    )

    df = df.merge(votes_sum, on='station_code_key', how='left')

    mask_good = df['ballots_good'].isna() & df['calculated_good_ballots'].notna()
    df.loc[mask_good, 'ballots_good'] = df.loc[mask_good, 'calculated_good_ballots']
    df.loc[mask_good, 'ballots_good_correction_reason'] = 'recovered_from_sum_of_vote_rows'
    df = append_validation_flag(df, mask_good, 'Recovered_Good_from_Votes')

    mask_used = (
        df['ballots_used'].isna()
        & df['ballots_good'].notna()
        & df['ballots_spoiled'].notna()
        & df['ballots_no_vote'].notna()
    )
    df.loc[mask_used, 'ballots_used'] = (
        df.loc[mask_used, 'ballots_good']
        + df.loc[mask_used, 'ballots_spoiled']
        + df.loc[mask_used, 'ballots_no_vote']
    )
    df.loc[mask_used, 'ballots_used_correction_reason'] = 'calculated_good_plus_spoiled_plus_no_vote'
    df = append_validation_flag(df, mask_used, 'Calculated_Used_via_Sum')

    mask_present = df['voters_present'].isna() & df['ballots_used'].notna()
    df.loc[mask_present, 'voters_present'] = df.loc[mask_present, 'ballots_used']
    df.loc[mask_present, 'voters_present_correction_reason'] = 'estimated_equal_to_ballots_used'
    df = append_validation_flag(df, mask_present, 'Estimated_Present_from_Used')

    df.drop(columns=['calculated_good_ballots'], inplace=True)
    return refresh_station_corrected_columns(df)


stations_df = heal_election_data(stations_df, votes_df)


In [ ]:
# Estimate eligible_voters only when it is missing, and keep the raw value + reason flag.
stations_df = ensure_station_audit_columns(stations_df)
known_data = stations_df.dropna(subset=['eligible_voters', 'voters_present'])
constituency_turnout_rate = (known_data['voters_present'] / known_data['eligible_voters']).replace([np.inf, -np.inf], np.nan).dropna().mean()
final_rate = constituency_turnout_rate if not np.isnan(constituency_turnout_rate) and constituency_turnout_rate > 0 else 0.71

mask = stations_df['eligible_voters'].isna() & stations_df['voters_present'].notna()
stations_df.loc[mask, 'eligible_voters'] = (stations_df.loc[mask, 'voters_present'] / final_rate).round()
stations_df.loc[mask, 'eligible_voters_correction_reason'] = f'estimated_from_turnout_rate_{final_rate:.4f}'
stations_df = append_validation_flag(stations_df, mask, f'ESTIMATED_ELIGIBLE_{final_rate:.2%}_TURNOUT')
stations_df = refresh_station_corrected_columns(stations_df)


In [72]:
stations_df.shape
stations_df.isnull().sum()

province             0
constituency_no      0
form_type            0
station_code         0
station_no           0
district             0
subdistrict          0
eligible_voters      0
voters_present       0
ballots_used         0
ballots_good         0
ballots_spoiled      0
ballots_no_vote      0
validation_flags    49
dtype: int64

In [ ]:
import os

output_path = os.path.join('..', 'data', 'clean_data')
os.makedirs(output_path, exist_ok=True)

# Keep one reviewed row per normalized station_code in the clean station outputs.
station_dupe_review = write_station_duplicate_review(stations_df, output_path)

station_file = os.path.join(output_path, '5_18_station.csv')
station_5_18 = select_best_station_rows(stations_df[stations_df['form_type'] == '5_18'].copy())
station_5_18.to_csv(station_file, index=False, encoding='utf-8-sig')

party_file = os.path.join(output_path, '5_18_party_station.csv')
station_5_18_party = select_best_station_rows(stations_df[stations_df['form_type'] == '5_18_party'].copy())
station_5_18_party.to_csv(party_file, index=False, encoding='utf-8-sig')

print('Files saved successfully in the root data/clean_data folder.')
print(f'5_18_station rows: {len(station_5_18)}')
print(f'5_18_party_station rows: {len(station_5_18_party)}')
print(f'Duplicate station review rows: {len(station_dupe_review)}')


In [ ]:
# Station lookup used for vote validation. Do not choose duplicates by largest ballots_good;
# use the same quality-based station policy as the clean station outputs.
station = select_best_station_rows(stations_df)


In [ ]:
def ensure_vote_audit_columns(df):
    df = add_station_code_key(df)
    df['votes'] = pd.to_numeric(df['votes'], errors='coerce')
    if 'votes_raw' not in df.columns:
        df['votes_raw'] = df['votes']
    if 'votes_correction_reason' not in df.columns:
        df['votes_correction_reason'] = ''
    if 'vote_validation_flags' not in df.columns:
        df['vote_validation_flags'] = ''
    df['votes_corrected'] = df['votes']
    return df


def append_vote_flag(df, mask, flag):
    existing = df.loc[mask, 'vote_validation_flags'].fillna('').astype(str)
    df.loc[mask, 'vote_validation_flags'] = existing.where(existing.eq(''), existing + '|') + flag
    return df


votes_df = ensure_vote_audit_columns(votes_df)
votes_df.drop_duplicates(inplace=True)
votes_df.dropna(subset=['station_code'], inplace=True)


In [ ]:
# Fix 1: Nullify physically impossible votes, but keep raw votes and a reason flag.
station_lookup = station[['station_code_key', 'ballots_good']].drop_duplicates('station_code_key')
votes_df = ensure_vote_audit_columns(votes_df)
votes_df = votes_df.merge(station_lookup, on='station_code_key', how='left', suffixes=('', '_station'))

impossible = (
    votes_df['votes'].notna()
    & votes_df['ballots_good'].notna()
    & (votes_df['votes'] > votes_df['ballots_good'])
)
print(f"Cleared {impossible.sum()} impossible vote values (votes > ballots_good)")
votes_df.loc[impossible, 'votes_correction_reason'] = 'cleared_votes_gt_ballots_good'
votes_df = append_vote_flag(votes_df, impossible, 'votes_gt_ballots_good_cleared')
votes_df.loc[impossible, 'votes'] = np.nan
votes_df['votes_corrected'] = votes_df['votes']
votes_df.drop(columns=['ballots_good'], inplace=True)

# Fix 2: Deduplicate votes from same station/entity/form with a review trail.
# Prefer non-review rows and non-missing votes; use vote size only as a final tie-breaker.
key_cols = ['station_code_key', 'entity_no', 'form_type']
before = len(votes_df)
votes_df['_duplicate_key_count'] = votes_df.groupby(key_cols)['station_code'].transform('size')
duplicate_vote_review = votes_df[votes_df['_duplicate_key_count'] > 1].copy()
duplicate_vote_review.to_csv(os.path.join(output_path, 'qa_duplicate_votes.csv'), index=False, encoding='utf-8-sig')

needs_review_rank = (
    votes_df['needs_review'].fillna(False).astype(bool)
    if 'needs_review' in votes_df.columns
    else pd.Series(False, index=votes_df.index)
)
votes_df['_needs_review_rank'] = needs_review_rank
votes_df['_vote_missing_rank'] = votes_df['votes'].isna()

votes_df = (
    votes_df
    .sort_values(
        key_cols + ['_needs_review_rank', '_vote_missing_rank', 'votes'],
        ascending=[True, True, True, True, True, False],
        kind='mergesort',
    )
    .drop_duplicates(subset=key_cols, keep='first')
    .drop(columns=['_duplicate_key_count', '_needs_review_rank', '_vote_missing_rank'], errors='ignore')
    .reset_index(drop=True)
)
print(f"Removed {before - len(votes_df)} duplicate vote rows -> {len(votes_df)} remaining")
print(f"Duplicate vote review rows: {len(duplicate_vote_review)}")


In [77]:
votes_df.shape
votes_df.isnull().sum()

province               0
constituency_no        0
form_type              0
station_code           0
station_no             0
district               0
subdistrict            0
entity_kind            0
entity_no              0
entity_name            0
party_name         15939
votes               3548
dtype: int64

In [78]:
# 5_18_votes will contain constituency/candidate data
votes_5_18 = votes_df[votes_df['form_type'] == '5_18'].copy()

# 5_18_party_vote will contain party-list data
party_5_18 = votes_df[votes_df['form_type'] == '5_18_party'].copy()

party_5_18.drop(columns=['party_name'], inplace=True)

votes_5_18.to_csv(os.path.join(output_path, '5_18_votes.csv'), index=False, encoding='utf-8-sig')
party_5_18.to_csv(os.path.join(output_path, '5_18_party_vote.csv'), index=False, encoding='utf-8-sig')

print(f"Files saved to: {os.path.abspath(output_path)}")
print(f"5_18_votes rows: {len(votes_5_18)}")
print(f"5_18_party_vote rows: {len(party_5_18)}")

Files saved to: d:\DSDE\ProjectDSDE_election2026\data\clean_data
5_18_votes rows: 2002
5_18_party_vote rows: 15939


In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer


def allocate_integer_budget(values, budget):
    values = pd.to_numeric(values, errors='coerce').fillna(0).clip(lower=0).astype(float)
    budget = max(int(budget), 0)
    if values.empty or budget == 0 or values.sum() <= 0:
        return pd.Series(0, index=values.index, dtype='int64')

    scaled = values * (budget / values.sum()) if values.sum() > budget else values
    floors = np.floor(scaled).astype(int)
    remainder = min(max(int(budget - floors.sum()), 0), len(floors))
    if remainder:
        fractions = (scaled - floors).sort_values(ascending=False)
        floors.loc[fractions.index[:remainder]] += 1
    return floors.astype('int64')


def add_imputation_flag(df, mask, flag):
    existing = df.loc[mask, 'imputation_flag'].fillna('').astype(str)
    df.loc[mask, 'imputation_flag'] = existing.where(existing.eq(''), existing + '|') + flag
    return df


def constrain_estimates_within_station(group):
    estimate_mask = group['votes_estimated'].fillna(False)
    ballots = pd.to_numeric(group['ballots_good'], errors='coerce').dropna()

    if ballots.empty:
        if estimate_mask.any():
            group.loc[estimate_mask, 'votes'] = pd.to_numeric(group.loc[estimate_mask, 'votes'], errors='coerce').fillna(0).clip(lower=0).round()
            group.loc[estimate_mask, 'votes_imputation_method'] = 'knn_estimated_no_ballots_good_constraint'
            group = add_imputation_flag(group, estimate_mask, 'estimated_without_ballots_good_constraint')
        return group

    ballots_good = max(int(np.floor(ballots.iloc[0])), 0)

    if estimate_mask.any():
        observed_sum = pd.to_numeric(group.loc[~estimate_mask, 'votes'], errors='coerce').fillna(0).clip(lower=0).sum()
        remaining = max(ballots_good - int(round(observed_sum)), 0)
        predicted = pd.to_numeric(group.loc[estimate_mask, 'votes'], errors='coerce').fillna(0).clip(lower=0, upper=ballots_good)
        target = min(int(round(predicted.sum())), remaining)
        constrained = allocate_integer_budget(predicted, target)

        group.loc[estimate_mask, 'votes'] = constrained
        group.loc[estimate_mask, 'votes_imputation_method'] = 'knn_estimated_constrained_to_station_ballots_good'
        group = add_imputation_flag(group, estimate_mask, 'estimated')
        if observed_sum > ballots_good:
            group = add_imputation_flag(group, estimate_mask, 'estimated_zero_observed_votes_exceed_ballots_good')
        elif int(round(predicted.sum())) > remaining:
            group = add_imputation_flag(group, estimate_mask, 'estimated_capped_to_remaining_ballots_good')

    final_votes = pd.to_numeric(group['votes'], errors='coerce').fillna(0).clip(lower=0)
    if final_votes.sum() > ballots_good:
        constrained_all = allocate_integer_budget(final_votes, ballots_good)
        group['votes'] = constrained_all
        observed_mask = ~estimate_mask
        group.loc[observed_mask, 'votes_imputation_method'] = 'observed_scaled_to_station_ballots_good'
        group = add_imputation_flag(group, pd.Series(True, index=group.index), 'station_total_scaled_to_ballots_good')

    return group
def constrained_knn_impute_votes(vote_path, station_path, output_path, estimated_output_path, label):
    vote_df = pd.read_csv(vote_path)
    station_df = pd.read_csv(station_path)

    vote_df = ensure_vote_audit_columns(vote_df)
    station_df = select_best_station_rows(station_df)
    station_features = station_df[['station_code_key', 'eligible_voters', 'ballots_good']].drop_duplicates('station_code_key')

    ml_df = vote_df.merge(station_features, on='station_code_key', how='left')
    for col in ['entity_no', 'eligible_voters', 'ballots_good', 'votes']:
        if col in ml_df.columns:
            ml_df[col] = pd.to_numeric(ml_df[col], errors='coerce')

    missing_before = int(ml_df['votes'].isna().sum())
    print(f"{label} missing BEFORE KNN: {missing_before}")

    features = ['entity_no', 'eligible_voters', 'ballots_good', 'votes']
    imputer = KNNImputer(n_neighbors=10, weights='uniform')
    imputed_data = imputer.fit_transform(ml_df[features])
    predicted_votes = pd.Series(imputed_data[:, features.index('votes')], index=ml_df.index).round().clip(lower=0)

    missing_mask = ml_df['votes'].isna()
    ml_df['votes_estimated'] = missing_mask
    ml_df['votes_prediction_knn'] = np.nan
    ml_df['votes_imputation_method'] = 'observed'
    ml_df['imputation_flag'] = ''
    ml_df.loc[missing_mask, 'votes_prediction_knn'] = predicted_votes.loc[missing_mask]
    ml_df.loc[missing_mask, 'votes'] = predicted_votes.loc[missing_mask]

    constrained_groups = [
        constrain_estimates_within_station(group.copy())
        for _, group in ml_df.groupby('station_code_key', dropna=False, sort=False)
    ]
    ml_df = pd.concat(constrained_groups, ignore_index=True) if constrained_groups else ml_df
    ml_df['votes'] = pd.to_numeric(ml_df['votes'], errors='coerce').round().astype('Int64')
    ml_df['votes_corrected'] = ml_df['votes']

    station_sums = ml_df.groupby('station_code_key')['votes'].sum(min_count=1).reset_index(name='vote_sum')
    check = station_sums.merge(station_features[['station_code_key', 'ballots_good']], on='station_code_key', how='left')
    over_budget = check[check['ballots_good'].notna() & (check['vote_sum'] > check['ballots_good'])]
    row_over_budget = ml_df[ml_df['ballots_good'].notna() & (ml_df['votes'] > ml_df['ballots_good'])]

    ml_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    ml_df.to_csv(estimated_output_path, index=False, encoding='utf-8-sig')

    print(f"{label} missing AFTER KNN: {int(ml_df['votes'].isna().sum())}")
    print(f"{label} station totals over ballots_good after constraints: {len(over_budget)}")
    print(f"{label} rows with votes > ballots_good after constraints: {len(row_over_budget)}")
    print(f"Saved constrained estimated votes to: {output_path}")
    print(f"Saved explicit estimated-layer copy to: {estimated_output_path}\n")
    return ml_df


In [ ]:
# ==========================================
# ESTIMATE PARTY LIST VOTES WITH CONSTRAINTS
# ==========================================

ml_party_df = constrained_knn_impute_votes(
    vote_path='../data/clean_data/5_18_party_vote.csv',
    station_path='../data/clean_data/5_18_party_station.csv',
    output_path='../data/clean_data/5_18_party_vote_KNNImputed.csv',
    estimated_output_path='../data/clean_data/5_18_party_vote_estimated.csv',
    label='Party votes',
)


In [ ]:
# ==========================================
# ESTIMATE CANDIDATE / CONSTITUENCY VOTES WITH CONSTRAINTS
# ==========================================

ml_candidate_df = constrained_knn_impute_votes(
    vote_path='../data/clean_data/5_18_votes.csv',
    station_path='../data/clean_data/5_18_station.csv',
    output_path='../data/clean_data/5_18_votes_KNNImputed.csv',
    estimated_output_path='../data/clean_data/5_18_votes_estimated.csv',
    label='Candidate votes',
)
